# Training-data simulations — visual inspection

**Purpose.** Every registered dataset is the output of a reaction–diffusion simulation, and
the entire inverse problem rests on the premise that those outputs really do carry Turing
patterns. This notebook re-runs those simulations where the payload makes that possible,
draws every registered sample, and measures whether each one is patterned — so the premise
is *checked* rather than assumed (CLAUDE.md §8: verify, don't assert).

**What it produces.** PNGs under `experiments/figures_report/training_data/`. That directory
is deliberately **tracked** — `.gitignore` un-ignores `experiments/figures_report/**/*.png`
because there the image *is* the deliverable, not a regenerable view of an array.

**Every field panel carries a colorbar, labelled axes, and tick marks in physical length
units** (the domain extent `L`), so a wavelength can be read straight off the figure. Those
three properties are asserted in `tests/test_td_figures.py`, not left to convention — the
pre-existing gallery `experiments/figures_report/stage0/f3_patterns_*.png` has colorbars but
no axis labels and no ticks, which is the gap this closes.

**What is and is not re-simulated.** Re-running a simulation needs the generating kinetics
*and* the simulation seed, because the initial condition is a seeded random perturbation of
the homogeneous steady state.

| family | kinetics stored | seed stored | re-simulatable |
|---|---|---|---|
| `three_gene_qvar`, `three_gene_multiL` | yes (`params_json`) | yes (`sim_seed`) | **yes — bit-exact** |
| `three_gene_train/test/val` | no | no | no — generator lived in gitignored `data/staging/` |
| `*_classical_*` | yes | no | no — and sub-family generators (`schnak_cross`, `gm_relay`, …) are not in `rngrn.data.rd_models` |

For the non-re-simulatable families the notebook plots the **stored** frames, which is what
training actually consumes. That distinction is stated on every figure rather than blurred.

**Runtime.** ~4 min end to end on CPU. The re-simulation section dominates (~11 s/sample).
No trainer is launched, so `scripts/guarded_run.sh` (CLAUDE.md §7a) does not apply; peak RSS
stays well under 1 GB because trajectories are loaded per sample rather than corpus-wide.

In [1]:
import os, sys, time, json
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))  if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(os.getcwd())
sys.path.insert(0, os.path.join(REPO, "scripts"))
sys.path.insert(0, os.path.join(REPO, "src"))

import td_figures as TD

DATASETS_ROOT = os.path.join(REPO, "data", "datasets")
OUT = os.path.join(REPO, "experiments", "figures_report", "training_data")
os.makedirs(OUT, exist_ok=True)

print("repo:  ", REPO)
print("data:  ", DATASETS_ROOT)
print("output:", OUT)

repo:   /home/benja/projects/personal/rngrn/worktrees/training-data-plots
data:   /home/benja/projects/personal/rngrn/worktrees/training-data-plots/data/datasets
output: /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data


## 1. Inventory — what is actually on disk

`payload.h5` is gitignored, so a fresh worktree *looks* provisioned (manifests are tracked)
but is not. If a dataset is missing here, run `bash scripts/link_payloads.sh`.

In [2]:
DATASETS = TD.available_datasets(DATASETS_ROOT)
print(f"{len(DATASETS)} datasets with a readable payload\n")

inventory = {}
rows = []
for ds in DATASETS:
    s = TD.load_samples(ds, DATASETS_ROOT)
    inventory[ds] = s
    rows.append(dict(
        dataset=ds, n=len(s), species=s[0]["n_species"], grid=s[0]["grid"],
        traj="yes" if s[0]["has_trajectory"] else "no",
        kinetics="yes" if s[0]["params"] else "no",
        resim="yes" if TD.is_resimulatable(s[0]) else "no",
        L_range=f"{min(x['L'] for x in s):.0f}-{max(x['L'] for x in s):.0f}",
    ))

hdr = f"{'dataset':28s} {'n':>4s} {'sp':>3s} {'grid':>5s} {'traj':>5s} {'kin':>4s} {'resim':>6s}  L range"
print(hdr); print("-" * len(hdr))
for r in rows:
    print(f"{r['dataset']:28s} {r['n']:4d} {r['species']:3d} {r['grid']:5d} "
          f"{r['traj']:>5s} {r['kinetics']:>4s} {r['resim']:>6s}  {r['L_range']}")
print("-" * len(hdr))
print(f"{'TOTAL':28s} {sum(r['n'] for r in rows):4d} samples")

11 datasets with a readable payload



dataset                         n  sp  grid  traj  kin  resim  L range
----------------------------------------------------------------------
three_gene_classical_test      16   3    96    no  yes     no  18-117
three_gene_classical_train     58   3    96    no  yes     no  18-150
three_gene_classical_val       11   3    96    no  yes     no  18-139
three_gene_multiL              92   3    96   yes  yes    yes  33-220
three_gene_qvar                34   3    96   yes  yes    yes  33-214
three_gene_test                20   3    96   yes   no     no  48-208
three_gene_train               88   3    96   yes   no     no  40-187
three_gene_val                 19   3    96   yes   no     no  40-139
two_gene_classical_test        13   2    96    no  yes     no  21-129
two_gene_classical_train       52   2    96    no  yes     no  18-101
two_gene_classical_val         10   2    96    no  yes     no  18-99
----------------------------------------------------------------------
TOTAL             

## 2. Is every sample actually patterned?

Two measurements per sample, both **image-only** (nothing here reads a generating parameter):

* **`cv`** — the spatial coefficient of variation of species 0, the generator's own
  accept/reject statistic: `gen_tg3.simulate_and_classify` discards any simulation with
  `cv < 0.05` as "collapsed to homogeneous". Reusing that exact number means the verdict
  here carries the meaning it had at generation time.

  > **This test is very nearly a tautology and must NOT be reported as evidence of
  > patterning.** Every generator in the corpus applies the same `cv < 0.05` reject rule,
  > so a corpus filtered at 0.05 having a minimum above 0.05 is simply what the filter
  > does. What it *does* establish is **payload integrity** — the cv recomputed from the
  > stored frame should reproduce the generator's stored `cv0` attribute, which is checked
  > below. The informative screen is `peak_bin`, in §2b.

  The key is called `has_contrast`, **not** `patterned`, deliberately: `eval/rollout.py`
  already owns `patterned` for a different quantity (rollout amplitude against
  `max(1e-3, 0.02*|x*_0|)`), and that one is pre-registered. The two are never comparable.
* **`k*_obs`** — the radially-averaged-power-spectrum peak, via `rngrn.observables.raps`,
  i.e. the same estimator the recovery objective uses.

* **`peak_bin`** — the RAPS bin holding the maximum. This answers a *different* question
  from `cv`, and keeping the two apart is the point: `cv` asks "is there contrast?", never
  "is the contrast **periodic**?". A field made of a few isolated blobs clears the `cv`
  rule comfortably while its spectrum decays monotonically from the lowest resolvable bin.
  A genuine Turing pattern peaks at an *interior* bin. Threshold `PEAK_BIN_MIN = 3`
  calibrated against all 413 samples — see §2b and `docs/DECISIONS.md` D-TDPLOT-1.

The stored `k_star` is then compared against `k*_obs`. That comparison is ground-truth-vs-
measurement and is legal *here* because this is a data-inspection notebook, not a recovery
criterion.

In [3]:
t0 = time.time()
records = []
stored_cv0 = {(ds, s["key"]): float(s["attrs"]["cv0"])
              for ds, samples in inventory.items() for s in samples if "cv0" in s["attrs"]}
for ds, samples in inventory.items():
    for s in samples:
        v = TD.patterning_verdict(s["final_frame"][0], s["L"])
        records.append(dict(dataset_id=ds, key=s["key"], morphology=s["morphology"],
                            L=s["L"], k_star=s["k_star"], **v))
print(f"measured {len(records)} samples in {time.time()-t0:.1f} s\n")

no_contrast = [r for r in records if not r["has_contrast"]]
print(f"cv < {TD.CV_PATTERNED_MIN}: {len(no_contrast)} / {len(records)}"
      f"   <- NEAR-TAUTOLOGICAL, the generator already rejected these")
for r in no_contrast:
    print(f"   {r['dataset_id']}/{r['key']}  cv={r['cv']:.4f}")

cv = np.array([r["cv"] for r in records])
print(f"cv over the corpus: min={cv.min():.4f}  median={np.median(cv):.3f}  max={cv.max():.3f}")

# The NON-circular part: does the cv recomputed from the stored frame reproduce the cv0
# the generator recorded at simulation time? This is a payload-integrity check.
dev = [abs(r["cv"] - stored_cv0[(r["dataset_id"], r["key"])])
       for r in records if (r["dataset_id"], r["key"]) in stored_cv0]
print(f"\nPAYLOAD INTEGRITY: |cv_recomputed - stored cv0| over {len(dev)}/{len(records)} "
      f"samples carrying a stored cv0: max = {max(dev):.2e}")

ko = np.array([r["k_star_obs"] for r in records])
kt = np.array([r["k_star"] for r in records])
ok = np.isfinite(ko) & np.isfinite(kt) & (kt > 0)
rel = np.abs(ko[ok] - kt[ok]) / kt[ok]
signed = (ko[ok] - kt[ok]) / kt[ok]
print(f"\n|k*_obs - k*_gen| / k*_gen over {ok.sum()} samples: "
      f"median={np.median(rel)*100:.1f}%  90th pct={np.percentile(rel,90)*100:.1f}%")
print(f"  SIGNED: median={np.median(signed)*100:+.1f}%, "
      f"{100*(signed>0).mean():.0f}% of samples above the generator's k* "
      f"-> a TENDENCY high, not a uniform offset")
print("\nmorphology classes:", dict(Counter(r["morphology"] for r in records)))

measured 413 samples in 0.2 s

cv < 0.05: 0 / 413   <- NEAR-TAUTOLOGICAL, the generator already rejected these
cv over the corpus: min=0.0633  median=0.688  max=2.882

PAYLOAD INTEGRITY: |cv_recomputed - stored cv0| over 413/413 samples carrying a stored cv0: max = 2.22e-07

|k*_obs - k*_gen| / k*_gen over 413 samples: median=8.3%  90th pct=22.7%
  SIGNED: median=+4.6%, 64% of samples above the generator's k* -> a TENDENCY high, not a uniform offset

morphology classes: {'spots': 225, 'stripes': 49, 'labyrinth': 139}


In [4]:
fig = TD.corpus_summary_figure(records)
TD.save(fig, OUT, "s0_corpus_patterning_summary.png")

  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/s0_corpus_patterning_summary.png  (0.18 MB)


'/home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/s0_corpus_patterning_summary.png'

### 2b. The periodicity screen — where the threshold comes from

`cv` and `peak_bin` are two different questions, and the corpus contains a sample where
they disagree. The peak-bin distribution below is what calibrates `PEAK_BIN_MIN`: if there
is a clean gap between the outliers and the bulk, the threshold belongs in that gap. If
there is not, the screen is not usable and must be reported as such.

In [5]:
pb = np.array([r["peak_bin"] for r in records])
print("RAPS peak-bin distribution over the corpus")
print(f"  min={pb.min()}  1st pct={np.percentile(pb,1):.0f}  5th={np.percentile(pb,5):.0f}  "
      f"median={int(np.median(pb))}  max={pb.max()}")
# NOT capped. An earlier version stopped at bin 9 and silently dropped the 45 samples at
# bins 10-14, so the printed table summed to 368 rather than 413.
hist = {int(b): int((pb == b).sum()) for b in range(pb.min(), pb.max() + 1)}
print("  full histogram (bin: count):", hist)
assert sum(hist.values()) == len(records), "the histogram must account for every sample"
print(f"  sums to {sum(hist.values())} = the whole corpus")
print(f"\n  threshold in use: PEAK_BIN_MIN = {TD.PEAK_BIN_MIN}")

nonper = [r for r in records if not r["periodic"]]
print(f"\nNOT PERIODIC (peak bin < {TD.PEAK_BIN_MIN}): {len(nonper)} / {len(records)}")
for r in nonper:
    print(f"   {r['dataset_id']}/{r['key']}  peak_bin={r['peak_bin']}  cv={r['cv']:.3f}  "
          f"periods across box={r['periods_across_box']:.2f}  "
          f"generator morphology='{r['morphology']}'")

if nonper:
    print("\n  ^ these clear the cv rule but their spectra have NO interior peak.")
    print("    Detail figures are written below so the verdict can be checked by eye.")
for r in nonper:
    s = [x for x in inventory[r["dataset_id"]] if x["key"] == r["key"]][0]
    fig = TD.detail_figure(s)
    TD.save(fig, OUT, f"x_NOT_PERIODIC__{r['dataset_id']}__{r['key']}.png")

RAPS peak-bin distribution over the corpus
  min=1  1st pct=3  5th=4  median=6  max=14
  full histogram (bin: count): {1: 1, 2: 0, 3: 14, 4: 44, 5: 78, 6: 155, 7: 49, 8: 12, 9: 15, 10: 14, 11: 8, 12: 9, 13: 11, 14: 3}
  sums to 413 = the whole corpus

  threshold in use: PEAK_BIN_MIN = 3

NOT PERIODIC (peak bin < 3): 1 / 413
   three_gene_qvar/sample_0032  peak_bin=1  cv=0.227  periods across box=2.46  generator morphology='spots'

  ^ these clear the cv rule but their spectra have NO interior peak.
    Detail figures are written below so the verdict can be checked by eye.


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/x_NOT_PERIODIC__three_gene_qvar__sample_0032.png  (0.12 MB)


## 3. Galleries — every registered sample, drawn

One contact sheet per dataset; one panel per sample, showing species 0 of the final frame.
Per-panel colorbars are deliberate: concentration ranges differ by orders of magnitude
across samples, so a single shared colorbar would flatten most panels to one colour.

Panel titles carry the sample key, the generator's morphology class, `L`, the generator's
`k*`, and the measured `cv`. A panel whose `cv` falls under the threshold is tagged
`NO CONTRAST`; a panel whose RAPS peak falls below `PEAK_BIN_MIN` is tagged `NOT PERIODIC`.

In [6]:
for ds, samples in inventory.items():
    n_sp = samples[0]["n_species"]
    fig = TD.gallery_figure(
        samples, ncols=8,
        title=(f"{ds} — {len(samples)} samples, species 0 of the final frame  "
               f"({n_sp}-species system, grid {samples[0]['grid']}x{samples[0]['grid']})"))
    # dpi 100, not the 130 used elsewhere: these sheets are the bulk of the tracked
    # figure payload, and 100 still gives ~340 px per 96x96 panel.
    TD.save(fig, OUT, f"g_{ds}.png", dpi=100)

  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/g_three_gene_classical_test.png  (0.57 MB)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/g_three_gene_classical_train.png  (1.93 MB)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/g_three_gene_classical_val.png  (0.42 MB)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/g_three_gene_multiL.png  (3.87 MB)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/g_three_gene_qvar.png  (1.26 MB)


...

  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/g_three_gene_train.png  (3.07 MB)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/g_three_gene_val.png  (0.70 MB)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/g_two_gene_classical_test.png  (0.44 MB)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/g_two_gene_classical_train.png  (1.87 MB)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/g_two_gene_classical_val.png  (0.33 MB)


## 4. Detail figures — all species plus the spectrum

The gallery shows *that* there is structure. The spectrum shows it has a single dominant
finite wavenumber — what a saturated Turing pattern looks like — rather than broadband noise
(which would rise towards k = 0) or a flat field.

A finite-wavenumber peak is **necessary, not sufficient**. "Turing-unstable" is a statement
about `sigma(k)` of the generating Jacobian, and is evaluated nowhere in this notebook.

One representative per (dataset, morphology class).

In [7]:
chosen = {}
for ds, samples in inventory.items():
    for s in samples:
        chosen.setdefault((ds, s["morphology"]), s)

print(f"{len(chosen)} (dataset, morphology) combinations\n")
for (ds, morph), s in sorted(chosen.items()):
    fig = TD.detail_figure(s)
    TD.save(fig, OUT, f"d_{ds}__{morph}__{s['key']}.png")

33 (dataset, morphology) combinations



  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/d_three_gene_classical_test__labyrinth__sample_0003.png  (0.21 MB)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/d_three_gene_classical_test__spots__sample_0000.png  (0.19 MB)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/d_three_gene_classical_test__stripes__sample_0001.png  (0.15 MB)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/d_three_gene_classical_train__labyrinth__sample_0002.png  (0.21 MB)


...

  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/d_two_gene_classical_train__spots__sample_0000.png  (0.15 MB)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/d_two_gene_classical_train__stripes__sample_0013.png  (0.10 MB)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/d_two_gene_classical_val__labyrinth__sample_0009.png  (0.15 MB)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/d_two_gene_classical_val__spots__sample_0000.png  (0.16 MB)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/d_two_gene_classical_val__stripes__sample_0001.png  (0.10 MB)


## 5. Trajectories — the pattern forming over time

The datasets that store a trajectory keep 6 frames spanning the second half of the
simulation. If the last frames are indistinguishable, the pattern has **saturated**, which
is what a converged Turing pattern should do.

In [8]:
traj_targets = [(ds, s) for ds, samples in inventory.items()
                for s in samples[:1] if s["has_trajectory"]]
print(f"{len(traj_targets)} datasets store trajectories\n")

for ds, s in traj_targets:
    full = TD.load_samples(ds, DATASETS_ROOT, limit=1, with_trajectory=True)[0]
    fig = TD.trajectory_figure(full)
    TD.save(fig, OUT, f"t_{ds}__{full['key']}.png")

5 datasets store trajectories



  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/t_three_gene_multiL__sample_0000.png  (0.20 MB)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/t_three_gene_qvar__sample_0000.png  (0.22 MB)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/t_three_gene_test__sample_0000.png  (0.28 MB)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/t_three_gene_train__sample_0000.png  (0.23 MB)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/t_three_gene_val__sample_0000.png  (0.27 MB)


## 6. Re-running the simulations

This is the section that literally **runs the training-data simulations** rather than
reading their output. For each re-simulatable sample the notebook rebuilds the generating
system from `params_json` + the stored `x*`, re-integrates the spectral IMEX scheme in
`scripts/gen_tg3.py` at the stored `sim_seed`, and compares against the frame in the
payload.

A **relative L2 error of 0** means the tracked generator plus the recorded seed fully
determine the training data — the dataset is exactly reproducible from version-controlled
code, which is what `gen_tg3.py` was written to guarantee (the previous generator used a
process-salted `abs(hash(...))` seed and did *not* reproduce).

In [9]:
resim_targets = []
for ds, samples in inventory.items():
    if TD.is_resimulatable(samples[0]):
        resim_targets += [(ds, s) for s in samples[:3]]

print(f"re-simulating {len(resim_targets)} samples "
      f"from {len({d for d,_ in resim_targets})} datasets\n")

resim_errors = []
for ds, s in resim_targets:
    t0 = time.time()
    out = TD.resimulate(s)
    dt = time.time() - t0
    fig, err = TD.resim_figure(
        stored=s["final_frame"][0], resim=out["final"][0], L=s["L"],
        title=(f"{ds} / {s['key']} — species 0 — re-simulated from params_json "
               f"at sim_seed={s['attrs']['sim_seed']}  ({dt:.1f} s)"))
    TD.save(fig, OUT, f"r_{ds}__{s['key']}.png")
    resim_errors.append((ds, s["key"], err, out["morphology"], s["morphology"]))
    print(f"   {ds}/{s['key']}: rel L2 err = {err:.3e}   "
          f"morphology {s['morphology']} -> {out['morphology']}   ({dt:.1f} s)")

print()
errs = np.array([e for _, _, e, _, _ in resim_errors])
print(f"reproduction error over {len(errs)} samples: max = {errs.max():.3e}")
print("EXACT reproduction" if errs.max() == 0 else "NOT exact - investigate")

re-simulating 6 samples from 2 datasets



  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/r_three_gene_multiL__sample_0000.png  (0.10 MB)
   three_gene_multiL/sample_0000: rel L2 err = 0.000e+00   morphology spots -> spots   (22.6 s)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/r_three_gene_multiL__sample_0001.png  (0.10 MB)
   three_gene_multiL/sample_0001: rel L2 err = 0.000e+00   morphology spots -> spots   (22.1 s)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/r_three_gene_multiL__sample_0002.png  (0.10 MB)
   three_gene_multiL/sample_0002: rel L2 err = 0.000e+00   morphology spots -> spots   (23.5 s)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/r_three_gene_qvar__sample_0000.png  (0.10 MB)
   three_gene_qvar/sample_0000: rel L2 err = 0.000e+00   morphology stripes -> stripes   (11.1 s)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/r_three_gene_qvar__sample_0001.png  (0.11 MB)
   three_gene_qvar/sample_0001: rel L2 err = 0.000e+00   morphology labyrinth -> labyrinth   (19.4 s)


  wrote /home/benja/projects/personal/rngrn/worktrees/training-data-plots/experiments/figures_report/training_data/r_three_gene_qvar__sample_0002.png  (0.10 MB)
   three_gene_qvar/sample_0002: rel L2 err = 0.000e+00   morphology spots -> spots   (20.3 s)

reproduction error over 6 samples: max = 0.000e+00
EXACT reproduction


## 7. What this notebook establishes, and what it does not

Fill in from the run above. Stated plainly, per CLAUDE.md §8.

**Established**
- **Payload integrity.** The cv recomputed from every stored final frame reproduces the
  generator's stored `cv0` attribute to the tolerance printed in §2.
- **The corpus is periodic on the spectral screen**, with exactly one exception (below).
- The wavenumber visible in each image tracks the one the generating system predicts:
  median relative deviation ~8%, 90th percentile ~23%, signed median ~+5% with ~64% of
  samples above the generator's `k*` — a *tendency* high, not a uniform offset. As far as
  this notebook can determine, **this is the first measurement of that bias on this
  corpus**. `src/rngrn/observables.py` asserts the same direction ("biased HIGH off onset")
  and a "~10-15%" reliability, but that docstring is unchanged since the initial template
  commit and cites `observables_spec.md`, which **does not exist in this repo** — so it
  corroborates the direction and calibrates nothing.

**NOT established, though an earlier draft of this notebook claimed it**
- "413/413 samples clear `cv >= 0.05`" is **circular**. Every generator in the corpus
  applies that same reject rule, so the result is what the filter does, not evidence about
  the data. It is retained above only as a payload-integrity check.
- `three_gene_qvar` and `three_gene_multiL` reproduce **bit-exactly** (relative L2 error
  0.000e+00) from `scripts/gen_tg3.py` plus the stored `sim_seed`.
- **One sample is not periodic.** `three_gene_qvar/sample_0032`, in a box of L = 177.8:
  ~95% of its pixels sit within 1% of the field's *dynamic range* above its minimum,
  species 1 and 2 are flat to `cv = 0.002`, and its spectrum has no interior peak. Visually
  it reads as 3 blobs; the payload's stored `n_components = 5` counts wrap-around fragments
  separately, and nothing in this notebook recomputes a component count. The generator
  labelled it `morphology='spots'` because `gen_tg3.classify` tests only
  `area_fraction < 0.34`, which an almost-empty field satisfies trivially. It is 1/413
  (0.24%) of the corpus and 1/34 (2.9%) of `three_gene_qvar`.

**Not established**
- *Nothing here is a recovery result.* This inspects the training **inputs** only. No model
  is fitted and no recovery metric is computed.
- "Patterned" and "Turing-unstable" remain different claims. This notebook measures the
  former from the image. The latter is a statement about `sigma(k)` of the generating
  Jacobian and is not evaluated here.
- The `three_gene_train/test/val` and `*_classical_*` families are **not** shown to be
  reproducible — only that their stored frames are patterned. Their generators are not in
  version control.
- The periodicity screen is a **coarse** instrument: it catches a spectrum with no interior
  peak, not a marginal pattern. `three_gene_classical_val/sample_0007` passes it (peak bin
  4) yet holds only 4 spots in the box; whether so few repeats is adequate for training is
  a separate question this notebook does not answer.
- No decision has been taken about excluding `sample_0032` from training. It is reported,
  not acted on.